# 03 — Review: Fix Buggy Analysis Code

This notebook is pure practice. Each section presents a buggy function with a description
of what it's supposed to do. Your job: find the bug and fix it.

These are the kinds of bugs that appear in real AI research analysis code.

## Setup

In [ ]:
import sys
sys.path.insert(0, "../../")
from src.checks import check_equal, check_type, check_contains

print("Setup complete.")

---
## Bug 1: Off-by-One in a Loop

**What it should do:** Count how many items in a list have a score above a threshold.
For `[0.9, 0.4, 0.8, 0.3, 0.7]` with `threshold=0.6`, the answer should be 3 (0.9, 0.8, 0.7).

**The bug:** The loop doesn't process all items.

In [ ]:
# Buggy code — run it to see the wrong result
def count_above_threshold(scores, threshold=0.6):
    count = 0
    for i in range(len(scores) - 1):  # Bug: -1 causes the last item to be skipped
        if scores[i] > threshold:
            count += 1
    return count

scores = [0.9, 0.4, 0.8, 0.3, 0.7]
result = count_above_threshold(scores)
print(f"Got: {result}, Expected: 3")

In [ ]:
# YOUR FIX HERE

def count_above_threshold_fixed(scores, threshold=0.6):
    count = 0
    # Fix the loop so it processes every item
    for score in scores:  # Iterate directly — no range() needed
        if score > threshold:
            count += 1
    return count

scores = [0.9, 0.4, 0.8, 0.3, 0.7]
result1 = count_above_threshold_fixed(scores)
print(f"Result: {result1}")

In [ ]:
check_equal(count_above_threshold_fixed([0.9, 0.4, 0.8, 0.3, 0.7]), 3,
            "Should count 3 items above threshold 0.6")
check_equal(count_above_threshold_fixed([0.1, 0.2], threshold=0.5), 0,
            "Should count 0 items above threshold 0.5")

---
## Bug 2: KeyError from Missing Dict Key

**What it should do:** Summarize model evaluation results. For any record without a `"version"` key,
it should use `"unknown"` as the version.

**The bug:** Directly accessing a key that might not be present.

In [ ]:
# Buggy code
def summarize_results(results):
    summary = []
    for r in results:
        summary.append({
            "id": r["id"],
            "score": r["score"],
            "version": r["version"],  # Bug: KeyError if 'version' is missing
        })
    return summary

results = [
    {"id": "r1", "score": 0.9, "version": "v2"},
    {"id": "r2", "score": 0.7},  # No 'version' key!
]

try:
    output = summarize_results(results)
except KeyError as e:
    print(f"KeyError: {e}")

In [ ]:
# YOUR FIX HERE

def summarize_results_fixed(results):
    summary = []
    for r in results:
        summary.append({
            "id": r["id"],
            "score": r["score"],
            "version": r.get("version", "unknown"),  # Fix: use .get() with default
        })
    return summary

results = [
    {"id": "r1", "score": 0.9, "version": "v2"},
    {"id": "r2", "score": 0.7},
]

output = summarize_results_fixed(results)
for row in output:
    print(row)

In [ ]:
check_equal(output[0]["version"], "v2", "First record version should be 'v2'")
check_equal(output[1]["version"], "unknown", "Missing version should default to 'unknown'")

---
## Bug 3: TypeError from String/Int Concatenation

**What it should do:** Build a summary string like `"Model gpt-4 scored 0.87 on task summarization"`.

**The bug:** Mixing types in string concatenation.

In [ ]:
# Buggy code
def build_summary(model, score, task):
    return "Model " + model + " scored " + score + " on task " + task
    # Bug: score is a float — can't concatenate float to str

try:
    msg = build_summary("gpt-4", 0.87, "summarization")
    print(msg)
except TypeError as e:
    print(f"TypeError: {e}")

In [ ]:
# YOUR FIX HERE — two good approaches

# Approach 1: f-string (recommended)
def build_summary_fstring(model, score, task):
    return f"Model {model} scored {score} on task {task}"

# Approach 2: str() conversion
def build_summary_str(model, score, task):
    return "Model " + model + " scored " + str(score) + " on task " + task

msg = build_summary_fstring("gpt-4", 0.87, "summarization")
print(msg)

msg2 = build_summary_str("gpt-4", 0.87, "summarization")
print(msg2)

In [ ]:
check_contains(msg, "gpt-4", "Summary should contain the model name")
check_contains(msg, "0.87", "Summary should contain the score")
check_contains(msg, "summarization", "Summary should contain the task")

---
## Bug 4: Logic Error — `=` Instead of `==`

**What it should do:** Filter a list of records to only those where `status == "pass"`.

**The bug:** In Python, `=` inside an `if` is a `SyntaxError`. This is actually a helpful feature!
Here we show what that looks like and fix it.

In [ ]:
# The following has a SyntaxError — shown as a comment to not halt the notebook.
# In a real file, you'd see this error immediately when trying to run it.

# def get_passing(records):
#     passing = []
#     for r in records:
#         if r["status"] = "pass":     # SyntaxError! = is assignment, not comparison
#             passing.append(r)
#     return passing

# Python's error message:
# SyntaxError: invalid syntax
#         if r["status"] = "pass":
#                        ^
# This is Python protecting you — assignment inside conditions is a common bug source.
# (Python 3.8+ added := for intentional assignment in conditions — the 'walrus operator')

print("The buggy version has a SyntaxError — see the comment above.")
print("Now implement the fixed version below.")

In [ ]:
# YOUR FIX HERE

def get_passing_fixed(records):
    passing = []
    for r in records:
        if r["status"] == "pass":  # Fixed: == for comparison
            passing.append(r)
    return passing

test_records = [
    {"id": 1, "status": "pass"},
    {"id": 2, "status": "fail"},
    {"id": 3, "status": "pass"},
    {"id": 4, "status": "fail"},
]

passing = get_passing_fixed(test_records)
passing_ids = [r["id"] for r in passing]
print(f"Passing IDs: {passing_ids}")

In [ ]:
check_equal(passing_ids, [1, 3], "Should return IDs 1 and 3 (status == 'pass')")

---
## Bug 5: Function Modifies List In-Place but Also Tries to Return It

**What it should do:** Sort a list of records by score (highest first) and return the sorted list.

**The bug:** `.sort()` modifies the list in-place and returns `None`. Returning the result of `.sort()`
returns `None`, not the sorted list.

This is a classic Python gotcha for JavaScript developers!
In JavaScript, `.sort()` returns the array. In Python, `.sort()` returns `None`.

In [ ]:
# Buggy code
def sort_by_score(records):
    return records.sort(key=lambda r: r["score"], reverse=True)
    # Bug: .sort() returns None! The sorted list exists, but we return None.

records = [
    {"id": 1, "score": 0.7},
    {"id": 2, "score": 0.9},
    {"id": 3, "score": 0.5},
]

result = sort_by_score(records)
print(f"Result: {result}")  # None! That's wrong.

In [ ]:
# YOUR FIX HERE — two approaches

records = [
    {"id": 1, "score": 0.7},
    {"id": 2, "score": 0.9},
    {"id": 3, "score": 0.5},
]

# Approach 1: sort in-place, then return the list
def sort_by_score_v1(records):
    records.sort(key=lambda r: r["score"], reverse=True)
    return records  # Return the list itself (now sorted)

# Approach 2: use sorted() which returns a NEW sorted list (non-mutating)
def sort_by_score_v2(records):
    return sorted(records, key=lambda r: r["score"], reverse=True)

sorted_records = sort_by_score_v2(records.copy())  # Use copy to preserve original
sorted_ids = [r["id"] for r in sorted_records]
print(f"Sorted IDs (highest score first): {sorted_ids}")
print(f"Scores: {[r['score'] for r in sorted_records]}")

In [ ]:
check_equal(sorted_ids, [2, 1, 3], "Should be sorted by score descending: IDs [2, 1, 3]")
check_type(sorted_records, list, "Result should be a list, not None")

---
## Bug 6: ZeroDivisionError When List Is Empty

**What it should do:** Compute the pass rate (fraction of records with `status == "pass"`).
When called with an empty list, it should return `None` (not crash).

**The bug:** Divides by `len(records)` without checking if the list is empty.

In [ ]:
# Buggy code
def compute_pass_rate(records):
    passed = sum(1 for r in records if r["status"] == "pass")
    return passed / len(records)  # Bug: ZeroDivisionError if records is empty

try:
    rate = compute_pass_rate([])  # Empty list!
    print(f"Pass rate: {rate}")
except ZeroDivisionError as e:
    print(f"ZeroDivisionError: {e}")

In [ ]:
# YOUR FIX HERE

def compute_pass_rate_fixed(records):
    if len(records) == 0:
        return None  # Can't compute a rate from empty data
    passed = sum(1 for r in records if r["status"] == "pass")
    return passed / len(records)

# Test with empty list
rate_empty = compute_pass_rate_fixed([])
print(f"Empty list result: {rate_empty}")  # Should be None

# Test with real data
test_records = [
    {"id": 1, "status": "pass"},
    {"id": 2, "status": "fail"},
    {"id": 3, "status": "pass"},
    {"id": 4, "status": "pass"},
]
rate_real = compute_pass_rate_fixed(test_records)
print(f"Pass rate: {rate_real:.2%}")  # Should be 75%

In [ ]:
check_equal(rate_empty, None, "Empty list should return None")
check_equal(rate_real, 0.75, "Pass rate should be 0.75 (3 out of 4)")

---
## Closing Thoughts

When you're doing AI safety research, you'll run analysis code on thousands of model outputs.
A subtle bug can silently produce wrong results.

Consider what each bug above could mean at scale:

- **Bug 1 (off-by-one):** Your analysis silently misses the last batch of results in every run
- **Bug 2 (KeyError):** Your pipeline crashes when it encounters older data with a different schema
- **Bug 3 (TypeError):** Your reporting script fails after processing 9,000 of 10,000 records
- **Bug 4 (= vs ==):** Caught immediately by Python — one of the language's safety features
- **Bug 5 (None return):** Your downstream code gets `None` instead of data and crashes confusingly
- **Bug 6 (ZeroDivision):** Your evaluation crashes on edge cases in production

These debugging habits — reading errors carefully, adding assertions, testing with small examples —
will save you from bad conclusions and wasted compute time.

**You've completed Module 03.** Proceed to [Module 04 — NumPy & Pandas Basics](../module_04_numpy_pandas/README.ipynb).